# HS4002 Week 5

## Basics of OLS Regression

Today we'll learn linear regression through one research question: **to what extent does job prestige predict income?**

We'll build three models:
- **Model 1**: income ~ prestige
- **Model 2**: income ~ prestige + gender
- **Model 3**: income ~ prestige + gender + age

In Python, `statsmodels` provides OLS regression with a **formula API** that mirrors R's `lm()` syntax almost exactly.

In [1]:
# !pip install pandas numpy plotnine statsmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

## Data Setup

In [2]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

# Model 1

Regress income (`income_cont`) on occupational prestige (`prestg10`).

Hint: `smf.ols('y ~ x', data=df).fit()` is the Python equivalent of `lm(y ~ x, data=df)` in R.

In [3]:
lm1 = smf.ols('income_cont ~ prestg10', data=df).fit()

## Print a summary of `lm1`

Equivalent to R's `summary(lm1)`.

In [4]:
print(lm1.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.175
Model:                            OLS   Adj. R-squared:                  0.174
Method:                 Least Squares   F-statistic:                     149.7
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           2.42e-31
Time:                        13:27:53   Log-Likelihood:                -8842.4
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     705   BIC:                         1.770e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.001e+04   9120.202     -2.194      0.0

### Provide a substantive interpretation of the results from `lm1`.

In [ ]:
# Your answer here
# This is a linear regression model that predicts income from prestige
# The dependent variable is income, it is a continuous variable, measure in dollars. One unit change in outcome variable correspond to $1 change in income.
# The independent variable is prestige, it is continuous variable, measure in points. One unit change in dependent variable correspond to 1 point change in prestige.
# We observe that on average one unit change in prestige is associated with 2237 dollars change in income (p<0.001), ceteris paribus.
# The null hypothesis of prestige coefficient is the true value of coef = 0; the alternative is coef /= 0.
# The p-value tell us if the true value of prestige coefficient is 0, the probability we can obtain coef=2237 or more extrem value is smaller than 0.001.
# The p-value is small, this is quite unlikely to happen null, thus we have sufficient evidence to reject null and infer the alternative hypothesis is likely to be true (r/=0).

# Model 2

Add gender (`sex_woman`) to the model. `statsmodels` automatically creates dummy variables for string/categorical columns.

In [6]:
lm2 = smf.ols('income_cont ~ prestg10 + sex_woman', data=df).fit()
print(lm2.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.180
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     77.45
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           3.96e-31
Time:                        13:56:57   Log-Likelihood:                -8840.2
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     704   BIC:                         1.770e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -1.381e+04   9556

In [ ]:
# This is a OLS linear regression model that predicts income from prestige and sex
# The dependent variable is income, it is a continuous variable, measure in dollars. One unit change in outcome variable correspond to $1 change in income.
# One of the independent variable is prestige, it is continuous variable, measure in points. One unit change in independent variable correspond to 1 point change in prestige.
    # We observe that on average one unit increase in prestige is associated with 2221 dollars increase in income (p<0.001), ceteris paribus.
    # The null hypothesis of prestige coefficient is the true value of coef = 0; the alternative is coef /= 0.
    # The p-value tell us if the true value of prestige coefficient is 0, the probability we can obtain coef=2221 or more extreme value is smaller than 0.001.
    # The p-value is small, this is quite unlikely to happen under null, thus we have sufficient evidence to reject null and infer there is linear association between income and prestige (coef/=0).
# Another independent variable is sex, it is a binary variable and men is the reference group.
    # We observe that on average, being a woman is associated with a $10440 decrease in annual income, compared to the reference group (men)(p=0.034), ceteris paribus.
    # The null hypothesis of sex coefficient is the true value of coef = 0; the alternative is coef /= 0.
    # The p-value tell us if the true value of pretige coefficient is 0, the probability we can obtain coed=-10440 or more extreme value is 0.034, which is smaller than 0.05.
    # The p-value is small, this is quite unlikely to happen under null, thus we have sufficient evidence to reject null, and infer that women's annual income differs from men's.

## Compare Model 1 and Model 2

### Comparing R² and Adjusted R² values

In [ ]:
comparison = pd.DataFrame({
	'Model':         ['Model 1', 'Model 2'],
	'R-Squared':     [lm1.rsquared,     lm2.rsquared],
	'Adj. R-Squared':[lm1.rsquared_adj, lm2.rsquared_adj]
})
comparison
# Model evaluation: based on the two measure, model is generately better

,Model,R-Squared,Adj. R-Squared
0,Model 1,0.175109,0.173939
1,Model 2,0.180349,0.178020


In [ ]:
# R-Squared compare model1 and model2, where 17.51% of sample variation in income has been explained by model1 and 18.03% of sample variation in income has been explained by model2. Model2's R-Squared is higher than model1's, thus model2 is a better model.
# However, R-squared can increase just simply adding in more variables, cannot conclude model2 is better with the R-squared alone.
# Thus, Adjust R-Squared give penalties to model2 because of adding additional variables sex. Thus, model's adjust R-Sqaured decrease to 17.80%, but still higher than model1 (17.39%).
# Therefore, model 2 is a better model compared to model1.

### Comparing AIC values

Lower AIC is better.

In [8]:
print(f'AIC Model 1: {lm1.aic:.2f}')
print(f'AIC Model 2: {lm2.aic:.2f}')
# Similar, but model 2 is slightly smaller

AIC Model 1: 17688.88
AIC Model 2: 17686.38


In [ ]:
# Model1 and Model2 have similar AIC values, AIC values give penalties for increase model complexivity by adding additional variables
# The smaller the AIC value is, the better the model.
# As model2 is slightly lower than model1, model is better than model1.

### Conducting an F-test

Equivalent to R's `anova(lm1, lm2)`.

This works only for **nested** models — Model 1's predictors must be a subset of Model 2's, fitted on the same rows. It cannot compare `prestige + gender` against `prestige + age`, since neither one contains the other. Remember, the null hypothesis tests if the additional variables in the unrestricted model **all** have coefficients of zero.


In [9]:
# anova_lm compares nested models with an F-test
print(anova_lm(lm1, lm2))

   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0     705.0  3.022575e+12      0.0           NaN       NaN       NaN
1     704.0  3.003374e+12      1.0  1.920049e+10  4.500653  0.034231


### Which model do you prefer? Justify using the statistics above.

In [ ]:
# Your answer here
# We estimate the F-test score is 4.5, with a p-value of 0.034.
# The null hypothesis is that the coefficient of the additional variable, sex, is zero; the alternative hypothesis coef of sex/=0.
# The p-value tell us that if the true value of sex coefficient is 0, the probability that we obtain F-statistic = 4.5 or more extreme value is 0.034, ceteris paribus.
# The p-value is small, this is unlikely to happen under null, thus we reject H0 and infer that the coefficient of sex is not 0 and adding sex improves the fit of model2 compared to model1.

# Model 3

Add age (`age`) to Model 2.

In [10]:
lm3 = smf.ols('income_cont ~ prestg10 + sex_woman + age', data=df).fit()
print(lm3.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.181
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     51.91
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           2.57e-30
Time:                        14:30:57   Log-Likelihood:                -8839.8
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     703   BIC:                         1.771e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -7449.5499   1.17

## Compare Model 2 and Model 3

In [11]:
comparison2 = pd.DataFrame({
	'Model':          ['Model 2', 'Model 3'],
	'R-Squared':      [lm2.rsquared,     lm3.rsquared],
	'Adj. R-Squared': [lm2.rsquared_adj, lm3.rsquared_adj],
	'AIC':            [lm2.aic,          lm3.aic]
})
comparison2

,Model,R-Squared,Adj. R-Squared,AIC
0,Model 2,0.180349,0.178020,17686.375978
1,Model 3,0.181360,0.177866,17687.503692


In [ ]:
# Although model3 has a slightly higher R-squared compared to model2, however, model3's adjusted R-square is lower than model2's.
# This is due to R-sqaured can simply increased by adding more variables to the model, while adjusted R-square and AIC give penalties for adding additional variables to increase model's complexivity.
# As model2 has a high adjusted r-squared and lower AIC value, model2 is a better model compared to model3.

In [12]:
print(anova_lm(lm2, lm3))

   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0     704.0  3.003374e+12      0.0           NaN       NaN       NaN
1     703.0  2.999671e+12      1.0  3.703232e+09  0.867886  0.351861


In [ ]:
# We estimate an f-statistic = 0.87, with a p-value of 0.35.
# The null hypothesis is that the true value of the additional variable's coefficient, age, is zero; the alternative is age coefficient is not zero.
# The p-value tell us: if the true value of age coefficient is 0, the probability that we obtain a F-statistics or more extreme value is 0.35.
# The p-value is quite big, this is quite likely to happen under null, thus we do not reject H0 as there is insufficient statistical evidence that age coefficients differ from 0.

# Making a professional-looking regression table

The Python `stargazer` package produces LaTeX/HTML tables equivalent to R's `stargazer` or `texreg`.

Run the cell below, then follow the Overleaf steps to turn that output into a typeset PDF table.

## How to use Overleaf

Overleaf is a LaTeX editor that runs in your browser. There is nothing to install, it compiles your document to PDF for you, and teammates can edit the same file at once. You'll be writing the research paper in it, so get comfortable now.

### 1. Set up your project

1. Go to [overleaf.com](https://www.overleaf.com) and register for a free account.
2. Open the course template: [https://www.overleaf.com/read/hqjkhwkfnfbr#d8258f](https://www.overleaf.com/read/hqjkhwkfnfbr#d8258f)
3. That link is **read-only**, so you can't type in it. Click **Menu → Copy Project** to get your own editable copy. (If that fails, click **New Project → Blank Project** and paste the template's text in yourself.)

### 2. Paste the table in

Copy the whole block the cell above printed — everything from `\begin{table}` down to `\end{table}` — and paste it between `\begin{document}` and `\end{document}`. Then click **Recompile**.

You'll have made your first LaTeX table!

In [14]:
from stargazer.stargazer import Stargazer

sg = Stargazer([lm1, lm2, lm3])
sg.title('Regression Results')

# LaTeX reads an underscore as a subscript, so the raw name 'income_cont'
# would stop the document from compiling in Overleaf.
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 1', 'Model 2', 'Model 3'], [1, 1, 1])
sg.covariate_order(['prestg10', 'sex_woman[T.Woman]', 'age', 'Intercept'])
sg.rename_covariates({
	'prestg10':           'Occupational prestige',
	'sex_woman[T.Woman]': 'Gender (woman = 1)',
	'age':                'Age',
	'Intercept':          'Intercept'
})

# Render as LaTeX — paste into Overleaf
print(sg.render_latex())

\begin{table}[!htbp] \centering
  \caption{Regression Results}
\begin{tabular}{@{\extracolsep{5pt}}lccc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{3}{c}{\textit{Dependent variable: income_cont}} \
\cr \cline{2-4}
\\[-1.8ex] & \multicolumn{1}{c}{Model 1} & \multicolumn{1}{c}{Model 2} & \multicolumn{1}{c}{Model 3}  \\
\\[-1.8ex] & (1) & (2) & (3) \\
\hline \\[-1.8ex]
 Occupational prestige & 2236.564$^{***}$ & 2221.175$^{***}$ & 2234.384$^{***}$ \\
& (182.823) & (182.515) & (183.082) \\
 Gender (woman = 1) & & -10442.232$^{**}$ & -10668.916$^{**}$ \\
& & (4922.158) & (4928.630) \\
 Age & & & -132.539$^{}$ \\
& & & (142.270) \\
 Intercept & -20009.515$^{**}$ & -13805.519$^{}$ & -7449.550$^{}$ \\
& (9120.202) & (9556.103) & (11742.412) \\
\hline \\[-1.8ex]
 Observations & 707 & 707 & 707 \\
 $R^2$ & 0.175 & 0.180 & 0.181 \\
 Adjusted $R^2$ & 0.174 & 0.178 & 0.178 \\
 Residual Std. Error & 65477.783 (df=705) & 65315.822 (df=704) & 65321.952 (df=703) \\
 F Statistic & 149.658$^{***}$

# Bonus Extension

Build a larger model (`lm_w`) using additional socioeconomic covariates available in the data. Compare its coefficient estimates and model-fit statistics to Models 1–3. Is it a better model? Justify your choice.

In [13]:
# Your answer here
lm_w = smf.ols(
    'income_cont ~ prestg10 + C(sex_woman) + age + C(degree)',
    data=df
).fit()

print(lm_w.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.266
Model:                            OLS   Adj. R-squared:                  0.259
Method:                 Least Squares   F-statistic:                     36.24
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           2.62e-43
Time:                        14:53:47   Log-Likelihood:                -8801.0
No. Observations:                 707   AIC:                         1.762e+04
Df Residuals:                     699   BIC:                         1.765e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept              -598.88

In [14]:
comparison_w = pd.DataFrame({
    'Model': ['Model 1', 'Model 2', 'Model 3', 'Model W'],
    'R-Squared': [
        lm1.rsquared,
        lm2.rsquared,
        lm3.rsquared,
        lm_w.rsquared
    ],
    'Adj. R-Squared': [
        lm1.rsquared_adj,
        lm2.rsquared_adj,
        lm3.rsquared_adj,
        lm_w.rsquared_adj
    ],
    'AIC': [
        lm1.aic,
        lm2.aic,
        lm3.aic,
        lm_w.aic
    ]
})

comparison_w

,Model,R-Squared,Adj. R-Squared,AIC
0,Model 1,0.175109,0.173939,17688.881424
1,Model 2,0.180349,0.178020,17686.375978
2,Model 3,0.181360,0.177866,17687.503692
3,Model W,0.266274,0.258926,17618.080573


In [15]:
print(anova_lm(lm1, lm2))
print(anova_lm(lm2, lm3))
print(anova_lm(lm3, lm_w))

   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0     705.0  3.022575e+12      0.0           NaN       NaN       NaN
1     704.0  3.003374e+12      1.0  1.920049e+10  4.500653  0.034231
   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0     704.0  3.003374e+12      0.0           NaN       NaN       NaN
1     703.0  2.999671e+12      1.0  3.703232e+09  0.867886  0.351861
   df_resid           ssr  df_diff       ss_diff          F        Pr(>F)
0     703.0  2.999671e+12      0.0           NaN        NaN           NaN
1     699.0  2.688526e+12      4.0  3.111446e+11  20.223908  8.895573e-16


In [ ]:
# Model W is better than Model1,2,3, as it has a larger R-squared and adjusted R-sqaure and smaller AIC value and p-value for its F-statistics.